<a href="https://colab.research.google.com/github/arundhatimsb/assignment/blob/main/sentence_embeddings_sa_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install torch matplotlib scikit-learn gdown sentence-transformers numpy scikit-learn --quiet

In [3]:
import gdown
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

In [8]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

folder_id = '1oEYY-Io_8XUBQZXnGftNjseU46Yxc5FK'
gdown.download_folder(
    id=folder_id,
    output='dataset',
    quiet=True,
    use_cookies=False
)

DATA_DIR = 'dataset'

df_train_sa = pd.read_csv(f'{DATA_DIR}/train_sa_10000.csv')
df_train_en = pd.read_csv(f'{DATA_DIR}/train_en_10000.csv')
df_dev_sa   = pd.read_csv(f'{DATA_DIR}/dev_sa_1000.csv')
df_dev_en   = pd.read_csv(f'{DATA_DIR}/dev_en_1000.csv')
df_test_sa  = pd.read_csv(f'{DATA_DIR}/test_sa_1000.csv')
df_test_en  = pd.read_csv(f'{DATA_DIR}/test_en_1000.csv')

print(df_test_sa)

KeyboardInterrupt: 

In [6]:
def merge_dataset(df_sa: pd.DataFrame,df_en: pd.DataFrame) -> pd.DataFrame:
  df = pd.merge(df_sa, df_en, on='Source_id', how='inner')
  df = df.dropna(subset=['Sentence_sa', 'Sentence_en']).reset_index(drop=True)
  print(f"{len(df)} aligned pairs loaded.")
  return df;

df_train = merge_dataset(df_train_sa,df_train_en)
df_test = merge_dataset(df_test_sa, df_test_en)
df_dev = merge_dataset(df_dev_sa, df_dev_en)

df_all = pd.concat([df_train, df_dev, df_test], ignore_index=True)
print(f"\nTotal pairs across all splits: {len(df_all)}")
df_all.head(3)


10000 aligned pairs loaded.
1000 aligned pairs loaded.
1000 aligned pairs loaded.

Total pairs across all splits: 12000


,Source_id,Sentence_sa,Sentence_en
0,1,"""Ctrl, S नुत्वा रक्षन्तु।""","Save it with Ctrl, S."
1,2,गुरुः छात्रान् एकवारं पाठयति ।,Teacher will teach the students only once.
2,3,चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित...,"To recreate this animation, I have to take two..."


In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'sentence-transformers/LaBSE'
model = SentenceTransformer(model_name, device=device)
print(f"Model loaded. Native embedding dimension: {model.get_sentence_embedding_dimension()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [1]:
def encode_sentences(sentences, model, batch_size=64, desc='Encoding'):
  embeddings = model.encode(
      sentences,
      batch_size = batch_size,
      show_progress_bar=True,
      convert_to_numpy=True,
      normalize_embeddings=True,
      device = device
  )
  return embeddings
print("Encoding sanskrit sentences.")
sa_sentences = df_all['Sentence_sa'].tolist()
emb_sa = encode_sentences(sa_sentences,model, desc='Sanskrit')

print("Encoding emglish sentences.")
en_sentences = df_all['Sentence_en'].tolist()
emb_en = encode_sentences(en_sentences,model, desc='English')
print(f"Raw Sanskrit embedding shape : {emb_sa.shape}")
print(f"Raw English  embedding shape : {emb_en.shape}")

In [9]:
target_dimension = 128
combined_emebedings = np.vstack([emb_sa, emb_en])
pca = PCA(n_components = target_dimension, random_state=seed)
pca.fit(combined_emebedings)

explained = pca.explained_variance_ratio_.sum() * 100
print(f"Cumulative explained variance in {target_dimension} components: {explained:.2f}%")

emb_sa_norm = normalize(pca.transform(emb_sa), norm='l2')
emb_en_norm = normalize(pca.transform(emb_en), norm='l2')

print(f"\n{'='*50}")
print(f"  EMBEDDING DIMENSION USED : {TARGET_DIM}")
print(f"{'='*50}")
print(f"Sanskrit embedding matrix shape : {emb_sa_norm.shape}")
print(f"English  embedding matrix shape : {emb_sa_norm.shape}")

NameError: name 'np' is not defined